In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

### 1. Montar Google Drive y Configurar Kaggle
Primero, conectamos Google Drive para acceder a tus archivos y configuramos las credenciales de Kaggle. Asegúrate de agregar tu `KAGGLE_USERNAME` y `KAGGLE_KEY` en la sección de Secretos de Colab (el icono de la llave a la izquierda).

In [3]:
from google.colab import drive
from google.colab import userdata
import os

# Montar Google Drive
drive.mount('/content/drive')

# Configurar credenciales de Kaggle desde los Secretos de Colab
# (Alternativamente, puedes subir tu archivo kaggle.json manualmente)
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Credenciales de Kaggle configuradas correctamente.")
except Exception as e:
    print("Aviso: No se encontraron las credenciales en los secretos de Colab.")
    print("Sube tu archivo kaggle.json o configura KAGGLE_USERNAME y KAGGLE_KEY en los secretos.")


Mounted at /content/drive
Credenciales de Kaggle configuradas correctamente.


### 2. Cargar el Modelo y los Datos de Prueba
Ajusta las variables `RUTA_AL_MODELO` y `RUTA_AL_TEST` con la ubicación exacta en tu Drive. El código asume el uso de `joblib` o `pickle`, muy común en modelos de Machine Learning tradicionales.

In [4]:
import pandas as pd
import joblib # Usado frecuentemente para guardar/cargar modelos de scikit-learn
import xgboost as xgb # Importante para cargar modelos XGBoost

# --- ¡CAMBIA ESTAS RUTAS POR LAS TUYAS! ---
RUTA_AL_MODELO = '/content/drive/My Drive/ML_Notebooks/F1 Spit Stops/Modelos/xgboost_optimizado_th065.joblib'
RUTA_AL_TEST = '/content/drive/My Drive/ML_Notebooks/F1 Spit Stops/test.csv'

print("Cargando datos de test...")
test_df = pd.read_csv(RUTA_AL_TEST)
display(test_df.head())

print("\nCargando modelo...")
# Al ser un archivo .joblib, la función joblib.load es la correcta
model = joblib.load(RUTA_AL_MODELO)
print("Modelo cargado con éxito.")


Cargando datos de test...


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0



Cargando modelo...
Modelo cargado con éxito.


### 3. Realizar Predicciones y Crear el Archivo de Envío (`submission.csv`)
Las competiciones de Kaggle suelen pedir un archivo con un formato específico (generalmente una columna de ID y una columna de la variable objetivo predicha).

In [6]:
# Definimos el nombre de la columna ID y el Target
COLUMNA_ID = 'id'
COLUMNA_TARGET = 'PitNextLap' # El target que Kaggle espera

# Separar los IDs para el archivo de envío final
ids = test_df[COLUMNA_ID]

# Eliminar las columnas que no se usaron en el entrenamiento
columnas_a_eliminar = ['id', 'Driver', 'Race', 'Year']
X_test = test_df.drop(columns=columnas_a_eliminar)

# --- CORRECCIÓN DEL ERROR ---
# XGBoost necesita que las columnas de texto sean de tipo 'category'
# Convertimos todas las columnas 'object' (texto) a 'category'
columnas_texto = X_test.select_dtypes(include=['object']).columns
for col in columnas_texto:
    X_test[col] = X_test[col].astype('category')
    print(f"Columna '{col}' convertida a tipo category.")

# Realizar predicciones
print("\nGenerando predicciones...")
predicciones = model.predict(X_test)

# Crear el DataFrame de envío (submission)
submission_df = pd.DataFrame({
    COLUMNA_ID: ids,
    COLUMNA_TARGET: predicciones
})

# Guardar a CSV (Kaggle normalmente exige que no tenga el índice de pandas)
submission_df.to_csv('submission.csv', index=False)

print("Vista previa de tu envío:")
display(submission_df.head())


Columna 'Compound' convertida a tipo category.

Generando predicciones...
Vista previa de tu envío:


,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,1


### 4. Enviar a Kaggle
Finalmente, usamos la API de línea de comandos de Kaggle para subir tu archivo `submission.csv` directamente a la competición.

In [7]:
# --- ¡CAMBIA ESTO POR EL NOMBRE DE LA COMPETICIÓN! ---

NOMBRE_DE_LA_COMPETICION = 'playground-series-s6e5'
MENSAJE_DE_ENVIO = 'Mi primera sumisión desde Colab'

# El signo '!' nos permite ejecutar comandos de terminal en Colab
!kaggle competitions submit -c {NOMBRE_DE_LA_COMPETICION} -f submission.csv -m "{MENSAJE_DE_ENVIO}"


401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload


In [8]:
from google.colab import files

print("Descargando submission.csv...")
files.download('submission.csv')


Descargando submission.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>